# Homework 9 — Serverless Deep Learning — ML Zoomcamp 2026 cohort

Run the 2026 hair classifier (`straight` vs not, one probability) with **ONNX Runtime**, first in Python, then inside a Lambda-compatible Docker container. Questions: `homework9_2026.md`. The model pair and the sample image are already in this folder; the setup cell checks them against `asset_manifest.json`.

**Submit at:** https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw09 · platform deadline **Mon 7 Dec**. Target: submit in the Tue 1 Dec 08:00 session.

> **Stack:** ONNX Runtime + plain Pillow, not the lecture's TF-Lite + keras-image-helper. The closest lecture material is the ONNX workshop (`../practice/09-serverless_commented.ipynb`, sections 9.3 – 9.5).

Scaffold only: question cells are empty. Q1 – Q4 run here; Q5 – Q6 need the terminal and Docker Desktop.

## Setup and assets

In [ ]:
import hashlib
import json
from io import BytesIO
from urllib import request

import numpy as np
from PIL import Image
import onnxruntime as ort

SAMPLE_URL = "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"

# Check the three frozen files against the checksums in asset_manifest.json.
manifest = json.load(open('asset_manifest.json'))
files = {
    'hair_classifier_v1.onnx': manifest['model']['sha256'],
    'hair_classifier_v1.onnx.data': manifest['external_data']['sha256'],
    'sample.jpeg': manifest['sample_image']['sha256'],
}
for name, expected in files.items():
    with open(name, 'rb') as f:
        actual = hashlib.sha256(f.read()).hexdigest()
    print('OK      ' if actual == expected else 'MISMATCH', name)

## Image preprocessing (pinned)

Verbatim from `homework9_2026.md`. Resize method, RGB conversion, channel order, normalisation and `float32` are part of the interface: don't swap in keras-image-helper or the workshop's `preprocess_pytorch_style` (they resize with `NEAREST`).

`Image.open('sample.jpeg').convert('RGB')` gives the same checksummed image as `download_image(SAMPLE_URL)`, if you'd rather work offline.

In [ ]:
def download_image(url):
    with request.urlopen(url, timeout=30) as response:
        return Image.open(BytesIO(response.read())).convert("RGB")


def prepare_image(image):
    image = image.resize((200, 200), Image.Resampling.BILINEAR)
    array = np.asarray(image, dtype=np.float32) / 255.0
    array = (array - np.array([0.485, 0.456, 0.406], dtype=np.float32)) / np.array(
        [0.229, 0.224, 0.225], dtype=np.float32
    )
    return np.transpose(array, (2, 0, 1))[None, ...]

## Q1. Look at the ONNX graph — output node name

```python
session = ort.InferenceSession("hair_classifier_v1.onnx", providers=["CPUExecutionProvider"])
```

Print `session.get_inputs()` and `session.get_outputs()` (name, shape, type of each).

> The model is **two files**: the 10 KB `.onnx` graph and the 80 MB `.onnx.data` weights. The `.data` file name is written inside the graph, so it must sit next to the `.onnx` with exactly that name; otherwise `InferenceSession` fails with *"cannot get file size ... hair_classifier_v1.onnx.data"*.

*(output / sigmoid / softmax / prediction)*

## Q2. Target size of `prepare_image`

Read it off the pinned function above (and check `prepare_image(...).shape` against the input shape from Q1). *(64x64 / 128x128 / 200x200 / 256x256)*

## Q3. Normalised input — first value of the R channel

`tensor = prepare_image(...)` on the sample image, then `tensor[0, 0, 0, 0]` rounded to 3 decimals (grader tolerance 0.01).

> The tensor is `(batch, channel, height, width)`: `[0, 0, 0, 0]` = first image, **R** channel, top-left pixel. Check `tensor.shape` first.
> A right Q3 does not prove the resize is right: the corner pixel can come out the same with `NEAREST` and `BILINEAR` while the rest of the image, and Q4, differ.

## Q4. Local ONNX inference — output probability

The pinned call, with the output name from Q1:

```python
output = session.run([output_name], {"input": prepare_image(download_image(SAMPLE_URL))})[0]
```

Report the probability rounded to 3 decimals (grader tolerance 0.02).

> `output` has shape `(1, 1)`: take the single value (`output.reshape(-1)[0]` or `output[0][0]`) before rounding.
> It is already a probability (sigmoid inside the model): don't apply sigmoid/softmax again, as you would to the clothing model's logits.
> On a test photo, the same model gave **0.79 with `BILINEAR` and 0.34 with `NEAREST`**: the resize method alone moved the answer across 0.5.

## Lambda handler — read it, run it locally

Read `lambda_function.py` (model loaded at module level, `prepare_image` as above, `lambda_handler` that accepts a direct event or an API Gateway event with a JSON-string `body`). Then, in this folder:

```bash
python smoke_test.py
```

> Event key is **`image_url`** (the lecture and workshop use `url`). The handler returns `{"statusCode": 200, "headers": ..., "body": "<JSON string>"}`.

## Q5. The Lambda configuration — base image

Open `Dockerfile` and read the `FROM` line (the part before `@sha256:` is the image name and tag). *(lambda/python:3.9 / lambda/python:3.11 / lambda/python:3.13 / python:3.13-slim-bookworm)*

> The digest after `@sha256:` pins the exact image build, so the image cannot drift between your build and the grader's.

## Q6. Invoke the container — `straight_probability`

Terminal 1, in this folder (Docker Desktop running):

```bash
docker build -t mlzoomcamp-2026-serverless .
docker run --rm -p 9000:8080 mlzoomcamp-2026-serverless
```

Terminal 2 (or the empty cell below):

```bash
curl -s -XPOST 'http://localhost:9000/2015-03-31/functions/function/invocations' \
  -H 'Content-Type: application/json' \
  -d '{"image_url":"https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"}'
```

Python skeleton to fill:

```python
import requests
url = 'http://localhost:____/2015-03-31/functions/function/invocations'
resp = requests.post(url, json={'image_url': SAMPLE_URL}, timeout=60).json()
# resp['body'] is a STRING -> parse it before reading straight_probability
```

Rounded to 3 decimals (tolerance 0.02). It must agree with Q4.

> **Port 9000**, not the lecture's 8080: `-p 9000:8080` maps host 9000 to the emulator's 8080.
> The response `body` is a JSON **string**: `json.loads(resp['body'])`, then the key.
> If Q6 and Q4 disagree, you are not running the same model, image or preprocessing (an old image tag? rebuild).
> On an Apple Silicon Mac the image builds as `arm64`: fine for this local test.

## Optional — deploy to AWS (not graded)

ECR repository → `docker tag` / `docker push` → Lambda *Container image* (architecture = the image's; timeout 30 s, memory 1024 MB) → API Gateway. Commands: section 9.6 of `../practice/09-serverless_commented.ipynb`.

> Skip this first if time runs short. If you do it, delete the function, the API and the ECR repository afterwards.